In [1]:
import sqlite3
import pandas as pd

con = sqlite3.connect(":memory:")   # a temporary database held in memory
for name in ["dim_course", "dim_student", "dim_result", "fact_enrollment"]:
    pd.read_csv(f"my_data_clean/{name}.csv").to_sql(name, con, index=False)

def sql(query):
    return pd.read_sql(query, con)   # runs a SQL query and shows the result as a table

sql("SELECT name FROM sqlite_master WHERE type = 'table'")

,name
0,dim_course
1,dim_student
2,dim_result
3,fact_enrollment


In [2]:
sql("""
SELECT f.EnrollmentID
FROM fact_enrollment f
LEFT JOIN dim_course  c ON f.CourseKey = c.CourseKey
LEFT JOIN dim_student s ON f.StudentID = s.StudentID
WHERE c.CourseKey IS NULL OR s.StudentID IS NULL
""")

,EnrollmentID


In [3]:
sql("""
SELECT COUNT(*) AS total_enrollments,
       ROUND(100.0 * SUM(CASE WHEN r.FinalResult = 'Withdrawn' THEN 1 ELSE 0 END) / COUNT(*), 1) AS withdrawal_rate_pct,
       ROUND(100.0 * SUM(CASE WHEN r.FinalResult <> 'Withdrawn' THEN 1 ELSE 0 END) / COUNT(*), 1) AS retention_rate_pct,
       ROUND(100.0 * SUM(CASE WHEN r.ResultGroup = 'Passed' THEN 1 ELSE 0 END) / COUNT(*), 1)    AS pass_rate_pct
FROM fact_enrollment f
JOIN dim_result r ON f.ResultKey = r.ResultKey
""")

,total_enrollments,withdrawal_rate_pct,retention_rate_pct,pass_rate_pct
0,32593,31.2,68.8,47.2


In [4]:
sql("""
SELECT c.Domain,
       COUNT(*) AS enrollments,
       ROUND(100.0 * SUM(CASE WHEN f.ResultKey = 4 THEN 1 ELSE 0 END) / COUNT(*), 1) AS withdrawal_rate_pct
FROM fact_enrollment f
JOIN dim_course c ON f.CourseKey = c.CourseKey
GROUP BY c.Domain
""")

,Domain,enrollments,withdrawal_rate_pct
0,STEM,21402,34.3
1,Social Sciences,11191,25.1


In [5]:
sql("""
SELECT c.ModuleCode,
       COUNT(*) AS enrollments,
       ROUND(100.0 * SUM(CASE WHEN f.ResultKey = 4 THEN 1 ELSE 0 END) / COUNT(*), 1) AS withdrawal_rate_pct,
       RANK() OVER (ORDER BY 1.0 * SUM(CASE WHEN f.ResultKey = 4 THEN 1 ELSE 0 END) / COUNT(*) DESC) AS risk_rank
FROM fact_enrollment f
JOIN dim_course c ON f.CourseKey = c.CourseKey
GROUP BY c.ModuleCode
ORDER BY risk_rank
""")

,ModuleCode,enrollments,withdrawal_rate_pct,risk_rank
0,CCC,4434,44.5,1
1,DDD,6272,35.9,2
2,FFF,7762,31.0,3
3,BBB,7909,30.2,4
4,EEE,2934,24.6,5
5,AAA,748,16.8,6
6,GGG,2534,11.5,7


In [6]:
sql("""
WITH by_intake AS (
    SELECT c.PresentationCode,
           100.0 * SUM(CASE WHEN f.ResultKey = 4 THEN 1 ELSE 0 END) / COUNT(*) AS withdrawal_rate
    FROM fact_enrollment f
    JOIN dim_course c ON f.CourseKey = c.CourseKey
    GROUP BY c.PresentationCode
)
SELECT PresentationCode,
       ROUND(withdrawal_rate, 1) AS withdrawal_rate_pct,
       ROUND(withdrawal_rate - LAG(withdrawal_rate) OVER (ORDER BY PresentationCode), 1) AS change_vs_previous
FROM by_intake
ORDER BY PresentationCode
""")

,PresentationCode,withdrawal_rate_pct,change_vs_previous
0,2013B,28.8,NaN
1,2013J,26.8,-2.0
2,2014B,33.5,6.7
3,2014J,34.0,0.5


In [7]:
sql("""
SELECT CASE WHEN ActiveAtCheckpoint = 'No' THEN 'Left before checkpoint'
            WHEN EarlyEngagement = 'Low' OR FirstAssessmentStatus <> '40+' THEN 'At risk'
            ELSE 'On track' END AS risk_flag,
       COUNT(*) AS enrollments,
       ROUND(100.0 * SUM(CASE WHEN ResultKey IN (3, 4) THEN 1 ELSE 0 END) / COUNT(*), 1) AS withdraw_or_fail_pct
FROM fact_enrollment
GROUP BY 1
ORDER BY 1
""")

,risk_flag,enrollments,withdraw_or_fail_pct
0,At risk,8810,68.8
1,Left before checkpoint,4953,100.0
2,On track,18830,32.9
